## Setup

Run the cell below to install the dependencies for this notebook.

In [1]:
# %pip install -r requirements.txt

## Load the data

In [2]:
%matplotlib inline
import csv, requests, os
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('data/df_2000_no_audio.csv')
print(f"Loaded {len(df)} rows")
df.head()

Loaded 10874 rows


,Title,Poet,Year,URL,Poem Text,pt_nopunct,str_len
0,I Could Be a Whale Shark,Aimee Nezhukumatathil,2018,https://poets.org/poem/i-could-be-whale-shark,"Bolinao, Philippines I am worried about tentac...",bolinao philippines i am worried about tentacl...,238.0
1,Throwing Children,Ross Gay,2023,https://poets.org/poem/throwing-children,It is really something when a kid who has a ha...,it is really something when a kid who has a ha...,281.0
2,The moon rose over the bay. I had a lot of fee...,Donika Kelly,2017,https://poets.org/poem/moon-rose-over-bay-i-ha...,"I am taken with the hot animal of my skin, gra...",i am taken with the hot animal of my skin grat...,203.0
3,“The world is a beautiful place”,Lawrence Ferlinghetti,2017,https://poets.org/poem/world-beautiful-place,The world is a beautiful place to be born into...,the world is a beautiful place to be born into...,264.0
4,Middle Passage,Robert Hayden,2017,https://poets.org/poem/middle-passage,"I Jesús, Estrella, Esperanza, Mercy: Sails fla...",i jesús estrella esperanza mercy sails flashin...,1210.0


# You are the classifier 👈


Based on the definitions provided, categorize the data you have been assigned as `Other Assault` or `Aggrevated Assault`.

In [4]:
df.columns.tolist()

['Title', 'Poet', 'Year', 'URL', 'Poem Text', 'pt_nopunct', 'str_len']

## LLM as the classifier 🤖

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
openrouter_key = os.getenv("OPENROUTER_API_KEY")

In [6]:
import diskcache
cache = diskcache.Cache('./cache')  # stores in ./cache folder

This is a **pydantic model**. It defines what format I want the output to come back in. It's for an LLM feature called "Structured Outputs", but also works with other LLM tools. This gets the LLM to always return structured json data, which gets read into python as a class.

In [7]:
from pydantic import BaseModel

# This is a pydantic model. It defines what format I want the output to come back in
# It's for an OpenAI feature called "Structured Output", but also works with other LLM tools
class Classification(BaseModel):
    llm: bool
    reason: str

#### Write Your System Prompt 👈

Edit `SYSTEM_PROMPT` in the cell below to tell the model how to classify each row. Also set `TEXT_COLUMN` to the name of the column you want to classify (default: `"Poem Text"`).

### Prompt 1:

>You will get a description of a crime from the LAPD. Your task is to classify it as either true (Aggravated Assault) or false (Other Assault). Use these descriptions of the two crime types to make your determination.
>
>Aggravated Assault: An unlawful attack by one person upon another for the purpose of inflicting severe or aggravated bodily injury. This type of assault usually is accompanied by the use of a weapon or by means likely to produce death or great bodily harm, including instances where the suspect struck the victim several times and/or knocked the victim unconcious. Weapons that indicate aggravated assault can include everday objects like glass bottles or other things that can cause injuries.
>
>Other Assault: Simple, Not Aggravated. Includes all assaults which do not involve the use of any weapon or dangerous object likeley to cause injuries and in which the victim did not sustain serious or aggravated injuries.
>
>Respond in JSON with two fields: "llm" (boolean) and "reason" (string).

### Prompt 2:
>You will get a description of a crime from the LAPD. Your task is to classify it as either true (Aggravated Assault) or false (Other Assault). Use these descriptions of the two crime types to make your determination.
>
>Aggravated Assault: An unlawful attack by one person upon another for the purpose of inflicting severe or aggravated bodily injury. This type of assault usually is accompanied by the use of a weapon or by means likely to produce death or great bodily harm, including instances where the suspect struck the victim several times and/or knocked the victim unconcious. Weapons that indicate aggravated assault can include everday objects like glass bottles or other things that can cause injuries, including cars. The crime report doesn't have to explicitly state the weapon used in order for it to be aggravated, as long as use of a weapon is implied by the facts. In order for an injury to make an assault aggravated, the severity of the injury should be the result of the action from the suspect, not a secondary effect. Use of a weapon doesn't necesasrily indicate aggravated assault if the victim does not seem to be seriously injured.
>
>Other Assault: Simple, Not Aggravated. Includes all assaults which do not involve the use of any weapon or dangerous object likeley to cause injuries and in which the victim did not sustain serious or aggravated injuries. Only brandishing a weapon without using to cause injury it isn't necessarily aggravated assault.
>
>Respond in JSON with two fields: "llm" (boolean) and "reason" (string).

In [8]:
SYSTEM_PROMPT = """
Classify each poem individually into categories: either "true" if it is about immigration or "false" if it is not about immigration.

These are poems of all kinds of styles and from all manner of authors (not only American authors) so it is not the case that all poems that include non-English words or non-American place names are about immigration (though it may be that some poems about immigration also do include non-English words or non_US place names).

Also note that a poem about moving or change is also not necessarily about immigration directly; you should be able to articulate a specific reason you think the poem is about immigration.

Respond in JSON with two fields: "llm" (boolean) and "reason" (string).
"""

# Set the column from your CSV that contains the text to classify
TEXT_COLUMN = "Poem Text"

MODEL = "google/gemini-2.5-flash-lite"

In [9]:
import json
import re
import time
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

def _parse_json_response(content):
    """Parse JSON from model response, handling markdown code fences."""
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass
    match = re.search(r'```(?:json)?\s*(.*?)\s*```', content, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    raise ValueError(f"Could not parse JSON from response: {content[:200]}")

@cache.memoize()
def ask_model_to_classify(text_description, model):
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text_description},
                ],
                response_format={"type": "json_object"},
                temperature=0,
            )
            content = response.choices[0].message.content
            parsed = _parse_json_response(content)
            if isinstance(parsed, list):
                parsed = parsed[0]
            Classification(**parsed)
            return json.dumps(parsed)
        except Exception as e:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

In [10]:
import json
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

WORKERS = 10  # parallel API requests; increase if you have high rate limits

def _classify_row(args):
    idx, text = args
    if pd.isna(text):
        return idx, None
    try:
        return idx, ask_model_to_classify(text, model=MODEL)
    except Exception as e:
        print(f"Row {idx} failed after retries: {e}")
        return idx, None

pairs = list(zip(df.index, df[TEXT_COLUMN]))
results = {}

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(_classify_row, p): p[0] for p in pairs}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying"):
        idx, result = future.result()
        results[idx] = result

df[MODEL] = pd.Series(results)
df['llm'] = df[MODEL].apply(lambda x: json.loads(x)['llm'] if x is not None else None)
df['reason'] = df[MODEL].apply(lambda x: json.loads(x)['reason'] if x is not None else None)
del df[MODEL]

df.head()

Classifying:   0%|          | 0/10874 [00:00<?, ?it/s]

Row 9452 failed after retries: Could not parse JSON from response: [
  {
    "llm": "false",
    "reason": "This poem uses geographical terms as puns for romantic or sexual innuendo. It does not discuss the movement of people between countries or the experience of be


,Title,Poet,Year,URL,Poem Text,pt_nopunct,str_len,llm,reason
0,I Could Be a Whale Shark,Aimee Nezhukumatathil,2018,https://poets.org/poem/i-could-be-whale-shark,"Bolinao, Philippines I am worried about tentac...",bolinao philippines i am worried about tentacl...,238.0,True,The poem explicitly mentions 'landed' and the ...
1,Throwing Children,Ross Gay,2023,https://poets.org/poem/throwing-children,It is really something when a kid who has a ha...,it is really something when a kid who has a ha...,281.0,False,The poem describes a heartwarming interaction ...
2,The moon rose over the bay. I had a lot of fee...,Donika Kelly,2017,https://poets.org/poem/moon-rose-over-bay-i-ha...,"I am taken with the hot animal of my skin, gra...",i am taken with the hot animal of my skin grat...,203.0,False,The poem describes a personal experience of be...
3,“The world is a beautiful place”,Lawrence Ferlinghetti,2017,https://poets.org/poem/world-beautiful-place,The world is a beautiful place to be born into...,the world is a beautiful place to be born into...,264.0,False,The poem reflects on the complexities and cont...
4,Middle Passage,Robert Hayden,2017,https://poets.org/poem/middle-passage,"I Jesús, Estrella, Esperanza, Mercy: Sails fla...",i jesús estrella esperanza mercy sails flashin...,1210.0,True,The poem explicitly details the horrors of the...


## Inspect results

In [11]:
# Distribution of classifications
df['llm'].value_counts()

llm
false    7233
False    2536
True      773
true      281
Name: count, dtype: int64

In [12]:
# Browse results — show the text column and classifications
pd.set_option('display.max_colwidth', None)
df[[TEXT_COLUMN, 'llm', 'reason']].head(3)

,Poem Text,llm,reason
0,"Bolinao, Philippines I am worried about tentacles. How you can still get stung even if the jelly arm disconnects from the bell. My husband swims without me—farther out to sea than I would like, buoyed by salt and rind of kelp. I am worried if I step too far into the China Sea, my baby will slow the beautiful kicks he has just begun since we landed. The quickening , they call it, but all I am is slow, a moon jelly floating like a bag in the sea. Or a whale shark. Yes—I could be a whale shark, newly spotted with moles from the pregnancy— my wide mouth always open to eat and eat with a look that says Surprise! Did I eat that much? When I sleep, I am a flutefish, just lying there, swaying back and forth among the kelpy mess of sheets. You can see the wet of my dark eye awake, awake. My husband is a pale blur near the horizon, full of adobo and not waiting thirty minutes before swimming. He is free and waves at me as he backstrokes past. This is how he prepares for fatherhood. Such tenderness still lingers in the air: the Roman poet Virgil gave his pet fly the most lavish funeral, complete with meat feast and barrels of oaky wine. You can never know where or why you hear a humming on this soft earth.",True,"The poem explicitly mentions 'landed' and the speaker's worry about her baby's movements slowing down after arriving in a new place ('Bolinao, Philippines'). This suggests a recent arrival, likely due to immigration, and the anxieties associated with it, such as adapting to a new environment and the physical changes of pregnancy in that context. The contrast between the speaker's slowness and her husband's freedom to swim further out also hints at the different experiences of navigating a new country."
1,It is really something when a kid who has a hard time becomes a kid who’s having a good time in no small part thanks to you throwing that kid in the air again and again on a mile long walk home from the Indian joint as her mom looks sideways at you like you don’t need to keep doing this because you’re pouring with sweat and breathing a little bit now you’re getting a good workout but because the kid laughs like a horse up there laughs like a kangaroo beating her wings against the light because she laughs like a happy little kid and when coming down and grabbing your forearm to brace herself for the time when you will drop her which you don’t and slides her hand into yours as she says for the fortieth time the fiftieth time inexhaustible her delight again again again and again and you say give me til the redbud tree or give me til the persimmon tree because she knows the trees and so quiet you almost can’t hear through her giggles she says ok til the next tree when she explodes howling yanking your arm from the socket again again all the wolves and mourning doves flying from her tiny throat and you throw her so high she lives up there in the tree for a minute she notices the ants organizing on the bark and a bumblebee carousing the little unripe persimmon in its beret she laughs and laughs as she hovers up there like a bumblebee like a hummingbird up there giggling in the light like a giddy little girl up there the world knows how to love.,False,"The poem describes a heartwarming interaction between an adult and a child during a walk home. While it touches on themes of joy, connection, and simple pleasures, there are no elements that suggest immigration, such as themes of displacement, cultural transition, or the experience of being an immigrant."
2,"I am taken with the hot animal of my skin, grateful to swing my limbs and have them move as I intend, though my knee, though my shoulder, though something is torn or tearing. Today, a dozen squid, dead on the harbor beach: one mostly buried, one with skin empty as a shell and hollow feeling, and, though the tentacles look soft, I do not touch them. I imagine they were startled to find themselves in the sun. I imagine the tide simply went out without them. I i

In [13]:
df.to_csv('data/df_llm_labels_1.csv', index=False)